In [ ]:
from src.api.client import ApiClient
from src.api.models import SystemPriceRecord
from src.data.clean import create_system_price_dataframe

In [ ]:
import httpx
import pandas as pd
import asyncio

In [ ]:
date = "2026-03-29"

In [ ]:
async with httpx.AsyncClient() as c:
    client = ApiClient(c)
    response = await client.fetch_system_prices(date)

In [ ]:
df = create_system_price_dataframe(response)

In [ ]:
rows = [sp.model_dump() for sp in response]

df = pd.DataFrame(rows)

In [ ]:
print(f"{1:05d}")

In [ ]:
dates = ["2025-03-10", "2022-08-01", "2025-10-26", "2026-01-02", "2026-04-19", "2026-03-29"]
async with httpx.AsyncClient() as c:
    client = ApiClient(c)
    tasks = [client.fetch_system_prices(d) for d in dates]

    results = await asyncio.gather(*tasks, return_exceptions=True)

    final_data = {}
    for d, res in zip(dates, results):
        if isinstance(res, Exception):
            print(f"Failed to fetch {d}: {res}")
        else:
            final_data[d] = pd.DataFrame([sp.model_dump() for sp in res])

In [ ]:
for k, v in final_data.items():
    try:
        print(k, len(v), v["systemSellPrice"].isna().any(), v["systemBuyPrice"].isna().any(), v["netImbalanceVolume"].isna().any())
    except Exception as err:
        print(k, err)

In [ ]:
final_data["2025-10-26"].info()

In [ ]:
from datetime import time, datetime, timedelta
from zoneinfo import ZoneInfo

tz = ZoneInfo("Europe/London")

_date = datetime(2025, 10, 26)

start = datetime.combine(_date, time.min, tzinfo=tz)
end = datetime.combine(_date + timedelta(days=1), time.min, tzinfo=tz)
print(int((end - start).total_seconds() // 1800))


start = datetime.combine(_date, time.min, tzinfo=ZoneInfo.no_cache("Europe/London"))
end = datetime.combine(_date + timedelta(days=1), time.min, tzinfo=ZoneInfo.no_cache("Europe/London"))
print(int((end - start).total_seconds() // 1800))